# 10 — Phase A envelope scan

**Purpose**: surface the *envelope* of what is observable from the 3-TPV mission within the 4250 km / 30 deg-threshold cost budget.

**Owner**: FPO-5 / Phase 0a / 0a.2 (FPO engineer).

What it answers:
1. **How many TPVs can be flown** in one mission (`n_max_tpvs`).
2. **How many chords** per TPV (`chord_max_per_tpv`).
3. **Two diverse plan bundles**:
   - **B1 (max TPVs)** — plans that hit the most TPVs.
   - **B2 (max chords on one TPV)** — plans that maximise survey depth on a single target.

**Feasibility-only policy (LOCK 1)**: every plan surfaced satisfies the 4250 km budget and routes around every restricted-airspace polygon.  Anything that would violate either is discarded before it reaches the Pareto archive.

**Loud fail**: missing input data raises immediately (no synthetic substitution).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from shapely.ops import unary_union

from multi_target_planner import (
    DEFAULT_DATA_DIR, TPV_FILENAME_PHASE0A,
    verify_data_sources,
    load_tpvs, load_restricted, load_atc, load_dropsonde,
    load_satellite_track, load_gatepoints, base_km,
    phase_a_envelope, format_envelope_summary,
)

report = verify_data_sources(DEFAULT_DATA_DIR)
missing = [(r, p) for r, (p, ok) in report.items() if not ok]
if missing:
    raise FileNotFoundError(f'Missing real-data inputs: {missing}')

tpvs, frame = load_tpvs(DEFAULT_DATA_DIR, TPV_FILENAME_PHASE0A)
restricted = load_restricted(DEFAULT_DATA_DIR, frame)
atc        = load_atc(DEFAULT_DATA_DIR, frame)
dropsonde  = load_dropsonde(DEFAULT_DATA_DIR, frame)
satellite  = load_satellite_track(DEFAULT_DATA_DIR, frame)
gatepoints = load_gatepoints(DEFAULT_DATA_DIR, frame)
base       = base_km(frame)
restricted_union = unary_union(restricted) if restricted else None
atc_union        = unary_union(atc) if atc else None

print(f'TPVs: {len(tpvs)}   gatepoints: {gatepoints.shape[0]}   restricted polys: {len(restricted)}')

In [ ]:
result = phase_a_envelope(
    tpvs=tpvs,
    base_km=base,
    gatepoints_km=gatepoints,
    restricted_union=restricted_union,
    atc_union=atc_union,
    # MVP grid; later phases will expand chord-count / angle-deviation choices.
    n_chord_options=(1, 2, 3, 4),
    angle_devs_deg=(-5.0, 0.0, 5.0),
    min_spacing_km=100.0,
)
print(format_envelope_summary(result, [t.label for t in tpvs]))

In [ ]:
def _draw_base_layers(ax, *, full=False):
    for poly in dropsonde:
        xs, ys = poly.exterior.xy
        ax.fill(xs, ys, color='#22bb55', alpha=0.10, zorder=2)
    for poly in atc:
        xs, ys = poly.exterior.xy
        ax.fill(xs, ys, color='#ffe066', alpha=0.30, zorder=3)
    for poly in restricted:
        xs, ys = poly.exterior.xy
        ax.fill(xs, ys, color='#ff6666', alpha=0.40, zorder=4)
        ax.plot(xs, ys, color='#cc0000', lw=0.8, zorder=4.5)
    tpv_colors = ['#1f77b4', '#d62728', '#2ca02c']
    for tpv, col in zip(tpvs, tpv_colors):
        verts = tpv.vertices_km
        closed = np.vstack([verts, verts[0]])
        ax.fill(verts[:, 0], verts[:, 1], color=col, alpha=0.20, zorder=5)
        ax.plot(closed[:, 0], closed[:, 1], color=col, lw=1.0, zorder=5.5)
        c = tpv.polygon_km.centroid
        ax.annotate(tpv.label, (c.x, c.y), color=col, fontweight='bold',
                    ha='center', va='center', fontsize=8, zorder=10)
    for i, (gx, gy) in enumerate(gatepoints):
        ax.plot(gx, gy, marker='D', color='#ff7f0e', ms=7, zorder=7)
    ax.plot(base[0], base[1], marker='*', color='black', ms=14, zorder=9)


def _plot_plan(ax, plan, *, title=''):
    _draw_base_layers(ax)
    pts = plan.route.polyline
    kinds = plan.route.seg_kinds
    for k, kind in enumerate(kinds):
        p1, p2 = pts[k], pts[k+1]
        if kind == 'chord':
            ax.plot([p1[0], p2[0]], [p1[1], p2[1]], '-', color='#d62728', lw=2.2, zorder=8)
        elif kind == 'spacer':
            ax.plot([p1[0], p2[0]], [p1[1], p2[1]], '-', color='#0066cc', lw=1.5, zorder=7)
        else:
            ax.plot([p1[0], p2[0]], [p1[1], p2[1]], '--', color='#333333', lw=1.0, alpha=0.8, zorder=7)
    all_x = np.concatenate([t.vertices_km[:, 0] for t in tpvs] + [[base[0]], gatepoints[:, 0]])
    all_y = np.concatenate([t.vertices_km[:, 1] for t in tpvs] + [[base[1]], gatepoints[:, 1]])
    pad = 500.0
    ax.set_xlim(all_x.min() - pad, all_x.max() + pad)
    ax.set_ylim(all_y.min() - pad, all_y.max() + pad)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=9)
    ax.grid(True, alpha=0.25)


def _plot_no_feasible(ax):
    _draw_base_layers(ax)
    ax.text(0.5, 0.5, 'no feasible plan', transform=ax.transAxes,
            ha='center', va='center', fontsize=14, color='#888888')
    all_x = np.concatenate([t.vertices_km[:, 0] for t in tpvs] + [[base[0]], gatepoints[:, 0]])
    all_y = np.concatenate([t.vertices_km[:, 1] for t in tpvs] + [[base[1]], gatepoints[:, 1]])
    pad = 500.0
    ax.set_xlim(all_x.min() - pad, all_x.max() + pad)
    ax.set_ylim(all_y.min() - pad, all_y.max() + pad)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.25)


bundles = [result.bundle_b1, result.bundle_b2]
max_per_bundle = max(1, max(len(b.plans) for b in bundles))
ncols = max_per_bundle
fig, axes = plt.subplots(2, ncols, figsize=(4.2 * ncols, 9), squeeze=False)
for row, bundle in enumerate(bundles):
    for col in range(ncols):
        ax = axes[row, col]
        if col < len(bundle.plans):
            plan = bundle.plans[col]
            visit = ', '.join(
                f'{tpvs[i].label}(x{cs.n_chords})'
                for i, cs in zip(plan.tpv_indices, plan.chord_choices)
            )
            title = (f'{bundle.name} #{col+1}\n'
                     f'n_tpvs={plan.n_tpvs}  total_chords={plan.total_chords}\n'
                     f'cost={plan.total_cost_km:.0f}/4250 km\n'
                     f'{visit}')
            _plot_plan(ax, plan, title=title)
        else:
            _plot_no_feasible(ax)
            ax.set_title(f'{bundle.name} #{col+1}\n(no further feasible plan)', fontsize=9)
    axes[row, 0].set_ylabel(f'Northing (km)\n{bundle.name}', fontsize=10)
axes[1, 0].set_xlabel('Easting (km)', fontsize=10)

legend_handles = [
    Line2D([0], [0], color='#d62728', lw=2.2, label='TPV survey chord'),
    Line2D([0], [0], color='#0066cc', lw=1.5, label='Inter-chord spacer'),
    Line2D([0], [0], color='#333333', lw=1.0, ls='--', label='Transit (auto-routed around restricted)'),
    Line2D([0], [0], marker='*', color='black', ms=10, ls='none', label='BASE'),
    Line2D([0], [0], marker='D', color='#ff7f0e', ms=7, ls='none', label='Gatepoint (mandatory)'),
    Patch(facecolor='#ff6666', alpha=0.40, label='Restricted airspace'),
    Patch(facecolor='#ffe066', alpha=0.30, label='ATC zone (Gander FIR, x1.35)'),
    Patch(facecolor='#22bb55', alpha=0.10, label='Dropsonde zones'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=4, fontsize=8,
           bbox_to_anchor=(0.5, -0.02), framealpha=0.95)

fig.suptitle(
    f'FPO-5 / Phase 0a / 0a.2 — Phase A envelope scan\n'
    f'n_max_tpvs = {result.n_max_tpvs}   '
    f'chord_max_per_tpv = {dict(zip([t.label for t in tpvs], result.chord_max_per_tpv))}   '
    f'{result.n_feasible:,} feasible / {result.n_enumerated:,} enumerated   '
    f'(scan {result.elapsed_sec:.0f} s)',
    fontsize=11,
)
plt.tight_layout(rect=(0, 0.04, 1, 0.96))
plt.savefig('10_phase_a_envelope.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved 10_phase_a_envelope.png')